In [38]:
import numpy as np
import pandas as pd 
import sklearn

In [39]:
df_rabat=pd.read_csv('/home/yazid/housing/scraping/csv/rabat.csv')
df_sale=pd.read_csv('/home/yazid/housing/scraping/csv/sale.csv')
df_temara=pd.read_csv('/home/yazid/housing/scraping/csv/temara.csv')
df_casa=pd.read_csv('/home/yazid/housing/scraping/csv/casa.csv')
df_marrakech=pd.read_csv('/home/yazid/housing/scraping/csv/marrakech.csv')
df_tanger=pd.read_csv('/home/yazid/housing/scraping/csv/tanger.csv')
df_kenitra=pd.read_csv('/home/yazid/housing/scraping/csv/kenitra.csv')
df_agadir=pd.read_csv('/home/yazid/housing/scraping/csv/agadir.csv')
df_fes=pd.read_csv('/home/yazid/housing/scraping/csv/fes.csv')
df_meknes=pd.read_csv('/home/yazid/housing/scraping/csv/meknes.csv')

In [40]:
df_rabat['Ville']='Rabat'
df_sale['Ville']='Sale'
df_temara['Ville']='Temara'
df_casa['Ville']='Casa'
df_marrakech['Ville']='Marrakech'
df_tanger['Ville']='Tanger'
df_kenitra['Ville']='Kenitra'
df_agadir['Ville']='Agadir'
df_fes['Ville']='Fes'
df_meknes['Ville']='Meknes'

In [41]:
len(df_rabat),len(df_sale),len(df_temara),len(df_casa),len(df_marrakech),len(df_tanger),len(df_kenitra),len(df_agadir),len(df_fes),len(df_meknes)

(940, 1410, 920, 1880, 1297, 1245, 1393, 1173, 1071, 1146)

In [42]:
df=pd.concat([df_rabat,df_sale,df_temara,df_casa,df_marrakech,df_tanger,df_kenitra,df_agadir,df_fes,df_meknes],ignore_index=True)

In [43]:
df.replace('Not Specified',np.nan,inplace=True)

,surface,chambres,sdb,etage,location,prix,Ville
0,171 m²,3 chambres,2 sdbs,Étage 4,"Rabat, Hay Riad",5 500 000,Rabat
1,140 m²,3 chambres,2 sdbs,NaN,"Rabat, Riyad",3 600 000,Rabat
2,122 m²,3 chambres,3 sdbs,Étage 3,"Rabat, Hay Riad",3 700 000,Rabat
3,138 m²,3 chambres,2 sdbs,Étage 2,"Hay Riad, Rabat",NaN,Rabat
4,137 m²,3 chambres,2 sdbs,Étage 2,"Hay Riad, Rabat",NaN,Rabat
...,...,...,...,...,...,...,...
12470,NaN,3 chambres,2 sdbs,Étage 1,"Autre secteur, Meknès",550 000,Meknes
12471,200 m²,4 chambres,3 sdbs,Rez de chaussée,"Autre secteur, Meknès",620 000,Meknes
12472,72 m²,3 chambres,1 sdb,Rez de chaussée,"Autre secteur, Meknès",320 000,Meknes
12473,128 m²,3 chambres,2 sdbs,Étage 2,"Hamria, Meknès",680 000,Meknes


In [44]:
numerical_cols=['surface','chambres','sdb','etage','prix']
df['etage']=df['etage'].replace({'Rez de chaussée':0})
df['surface']=df['surface'].astype(str).replace(',','.',regex=False).str.extract(r'(\d+\.?\d*)')[0].astype(float)
df['chambres']=df['chambres'].astype(str).str.extract(r'(\d+)')[0].astype(float)
df['sdb']=df['sdb'].astype(str).str.extract(r'(\d+)')[0].astype(float)
df['etage']=df['etage'].astype(str).str.extract(r'(\d+)')[0].astype(float)
df['prix']=df['prix'].astype(str).str.replace(r'[\s\xa0]','',regex=True).str.extract(r'(\d+)')[0].astype(float)

In [45]:
df['surface'].max()

np.float64(665380000.0)

In [46]:
df.dropna(subset=['prix'],inplace=True)

In [47]:
df['location']=df['location'].fillna(df['Ville'])

Let's start with some feature engineering:

In [48]:
df['etage']=df['etage'].round().astype(float)
df['est rez de chausee']=(df['etage']==0)
df['surface par chambre']=df['surface']/(df['chambres']+1)
df['2 sdb+']=(df['sdb']>=2).astype(bool)
df['prix m2']=df['prix']/df['surface']

In [49]:
df['Ville'].value_counts()

Ville
Casa         1727
Kenitra      1254
Marrakech    1203
Sale         1178
Tanger       1084
Agadir        949
Meknes        938
Fes           861
Rabat         822
Temara        804
Name: count, dtype: int64

In [50]:
(df.isna().sum()/len(df))*100

surface                31.099815
chambres               15.425139
sdb                    17.754159
etage                  16.469501
location                0.000000
prix                    0.000000
Ville                   0.000000
est rez de chausee      0.000000
surface par chambre    31.293900
2 sdb+                  0.000000
surface par sdb        31.802218
prix m2                31.099815
dtype: float64

In [51]:
df['surface'].quantile([0.80,0.85,0.90,0.95,0.99,0.995,0.975,0.9999])

0.8000    1.260000e+02
0.8500    1.380000e+02
0.9000    1.540000e+02
0.9500    1.830000e+02
0.9900    3.000000e+02
0.9950    3.660000e+02
0.9750    2.160000e+02
0.9999    1.788611e+08
Name: surface, dtype: float64

In [52]:
df['prix m2'].quantile([0.80,0.85,0.90,0.95,0.975,0.999])

/home/yazid/housing/.venv/lib64/python3.14/site-packages/numpy/lib/_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


0.800    15294.995610
0.850    17725.974026
0.900    20952.738185
0.950    26639.053254
0.975    44059.808612
0.999             NaN
Name: prix m2, dtype: float64

In [53]:
df['prix'].quantile([0.9,0.95,0.975,0.99,0.9925,0.995,0.9975,0.999])

0.9000    2.460000e+06
0.9500    3.300900e+06
0.9750    4.626250e+06
0.9900    1.681000e+07
0.9925    4.085750e+07
0.9950    7.452500e+07
0.9975    1.400000e+08
0.9990    4.762900e+08
Name: prix, dtype: float64

Based on the quantiles we keep only relevant rows:

In [54]:
df=df[(df['prix']<9.5e+06) &(df['surface']<4.3e+03)]
df=df[(df['prix m2']>=2500) &(df['prix m2']<=40000)]

In [55]:
#We drop prix par m2 to not have data leakage:
df.drop('prix m2',axis=1,inplace=True)

Let's extract the neighborhood name:

In [56]:
cities_pattern = r"\b(Rabat|Casablanca|Casa|Salé|Sale|Témara|Temara|Marrakech)\b"
df['location']=df['location'].str.replace(cities_pattern,'',regex=True,case=False).str.replace(r'[^\w\s]','',regex=True).str.strip()

In [57]:
df['location']=df['location'].replace(r'^\s*$',np.nan,regex=True).str.replace('Autre secteur','Autre Secteur').fillna('Autre Secteur')

In [58]:
df['location']=np.where(df['location']=='Autre Secteur','Autre Secteur_'+df['Ville'].astype(str),df['location'])

In [59]:
df['location'].value_counts()

location
Autre Secteur Meknès        222
Guéliz                      167
Hay Mohammadi Agadir        138
Agdal                       104
alliance Kénitra            104
                           ... 
Dar Mehrez Fès                1
Mont Fleuri 2 Fès             1
Hay Ouifak Fès                1
Quartier Ben Slimane Fès      1
Douh Fès                      1
Name: count, Length: 393, dtype: int64

In [60]:
df.columns.tolist()

['surface',
 'chambres',
 'sdb',
 'etage',
 'location',
 'prix',
 'Ville',
 'est rez de chausee',
 'surface par chambre',
 '2 sdb+',
 'surface par sdb']

In [61]:
df['log prix']=np.log1p(df['prix'])
df.drop('prix',axis=1,inplace=True) #So we get rid of right skewed distribution

In [62]:
num_col=['surface','chambres','sdb','etage']
for col in num_col:
    df[col]=df[col].astype(float)

In [63]:
from sklearn.model_selection import train_test_split
X=df.drop('log prix',axis=1)
y=df['log prix']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
loc_counts=X_train['location'].value_counts()
good_locs=loc_counts[loc_counts>=5].index
X_train['location']=X_train['location'].where(X_train['location'].isin(good_locs),'Autre Secteur'+df['Ville'])
X_test['location']=X_test['location'].where(X_train['location'].isin(good_locs),'Autre Secteur'+df['Ville'])


In [64]:
from sklearn.impute import SimpleImputer
num_cols=X_train.select_dtypes(include=['number']).columns
cat_cols=X_train.select_dtypes(exclude=['number']).columns
imputer=SimpleImputer(strategy='median')
X_train_num_imputed=imputer.fit_transform(X_train[num_cols])
X_test_num_imputed=imputer.transform(X_test[num_cols])
X_train_imputed_df=pd.DataFrame(X_train_num_imputed,columns=num_cols,index=X_train.index)
X_test_imputed_df=pd.DataFrame(X_test_num_imputed,columns=num_cols,index=X_test.index)
X_train_imputed=pd.concat([X_train_imputed_df,X_train[cat_cols]],axis=1)
X_test_imputed=pd.concat([X_test_imputed_df,X_test[cat_cols]],axis=1)

Lets build a baseline model:

In [65]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,TargetEncoder,StandardScaler
one_hot=['Ville','est rez de chausee','2 sdb+']
num_col=['surface','chambres','sdb','etage']
tar_encode=['location']

In [66]:
preprocessor=ColumnTransformer([('onehot',OneHotEncoder(handle_unknown='ignore'),one_hot),('target',TargetEncoder(smooth=10),tar_encode),('standard',StandardScaler(),num_col)])
X_train_proc=preprocessor.fit_transform(X_train_imputed,y_train)
X_test_proc=preprocessor.transform(X_test_imputed)

Let's build our first model

Let's try GridSearchCv

Let's try RandomizedSearch

In [67]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
param_dist={'max_depth':[6,12,20,25,20,40,None],
           'min_samples_leaf':[1,2,5,10,20,40],
           'max_features':['sqrt','log2',0.5,0.8]}
randomsearch=RandomizedSearchCV(estimator=RandomForestRegressor(n_estimators=300,random_state=42,
                                                               n_jobs=-1),
                               param_distributions=param_dist,
                               n_iter=20,
                               cv=5,
                               scoring='neg_mean_absolute_error',
                               n_jobs=-1)
randomsearch.fit(X_train_proc,y_train)
print(randomsearch.best_params_)

{'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 40}


Based on these results let's make model2:

In [74]:
model_2=RandomForestRegressor(n_estimators=500,max_depth=40,min_samples_leaf=1,random_state=42,n_jobs=-1,max_features=0.8)
model_2.fit(X_train_proc,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",40
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.8
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max

In [75]:
from sklearn.metrics import mean_absolute_error
y_pred_train_2=model_2.predict(X_train_proc)
mae_2_train=mean_absolute_error(y_pred_train_2,y_train)
mae_2_train

0.07201863062463877

In [76]:
y_pred_2=model_2.predict(X_test_proc)
mae_2=mean_absolute_error(y_pred_2,y_test)
mae_2,df['log prix'].median()

(0.2809354052852696, np.float64(13.652992804936396))

Let's try histgradientboosting

In [84]:
from sklearn.ensemble import HistGradientBoostingRegressor
X_train_hgb=X_train_imputed.copy()
X_test_hgb=X_test_imputed.copy()
for col in ['Ville','location','2 sdb+','est rez de chausee']:
    X_train_hgb[col]=X_train_hgb[col].astype('category')
    X_test_hgb[col]=X_test_hgb[col].astype('category')
hgb=HistGradientBoostingRegressor(max_iter=500,max_depth=8,learning_rate=0.05,min_samples_leaf=30,categorical_features=['Ville','location','2 sdb+','est rez de chausee'])
hgb.fit(X_train_hgb,y_train)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",500
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",8
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",30
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide <categorical_support_gbdt>` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.","['Ville', 'location', ...]"
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, 

In [86]:
y_pred_hgb_1=hgb.predict(X_test_hgb)
mae_hgb_1=mean_absolute_error(y_pred_hgb_1,y_test)
mae_hgb_1

0.29119828938282993